# Liu2024 S-JEPA Embeddings + Classical Probe — Fixed Preprocessing + S-JEPA-style 5-fold CV

**Goal:** Test whether S-JEPA PreLocal embeddings are useful when the unstable neural classification head is replaced with a small classical classifier (shrinkage LDA or L2-logistic).

**Important fixes in this version:**
- Fixed the trial/channel/time reshape bug in preprocessing. Liu2024 data are `trials × channels × time`; when concatenating trials for MNE, the code now uses `channels × trials × time`, not a raw reshape that scrambles EEG channels.
- Default CV is now **S-JEPA-style 5-fold within-subject stratified CV**: 5 folds per subject, roughly 32 train / 8 test trials per fold.
- Kept the original Liu-style repeated 60/40 holdout available through `CONFIG["cv_scheme"] = "liu_repeated_holdout"`.
- Added fold diagnostics: train accuracy, train balanced accuracy, train/test class counts, PCA component count, and PCA explained variance sum.

**Leakage controls:**
- Preprocessing uses fixed transforms only.
- Spatial-conv fine-tuning uses only training fold trials.
- PCA is fit on training fold embeddings only.
- Classifier is fit on training fold features only.
- Nothing from the test fold is seen during fitting.


## 1. Imports

In [1]:
import os
import re
import sys
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from scipy.io import loadmat
from scipy import signal as sp_signal

import mne
mne.set_log_level("WARNING")

from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

from braindecode.models import SignalJEPA_PreLocal

warnings.filterwarnings('ignore', category=RuntimeWarning)
print(f"torch {torch.__version__}, numpy {np.__version__}")

def resolve_device(cfg_device="auto"):
    if cfg_device != "auto":
        return torch.device(cfg_device)
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

/home/vegorov/Repos/eeg_jepa_research/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.10.0+cu128, numpy 2.4.3


## 2. CONFIG

In [2]:
CONFIG = {
    # --- Paths ---
    "data_root":             "../../liu2024_data/liu2024_figshare/sourcedata",
    "artifact_root":         "../../artifacts/liu2024_sjepa_embeddings_lda_sjepa5fold_fixed",
    # S-JEPA checkpoint: set to None to use HuggingFace from_pretrained.
    # For your Liu2024-pretrained model, point this to the local student/backbone checkpoint.
    "sjepa_checkpoint_path": None,
    "sjepa_repo_id":         "braindecode/signal-jepa_without-chans",

    # --- Dataset ---
    "subjects":   "all",
    "random_state": 2026,
    "sfreq_raw":  500,
    "sfreq_model": 128,
    # MI cue/marker is around 2.0 s in the 8 s trial. This 4.2 s model window runs 1.5-5.7 s.
    "mi_window_s": (1.5, 5.7),
    "bandpass_hz": (0.5, 40.0),

    # --- S-JEPA window ---
    # S-JEPA PreLocal expects this many samples at sfreq_model Hz.
    # At 128 Hz, 4.195 s ≈ 537 samples.
    "window_samples": 537,

    # --- Cross-validation ---
    # Default: closer to the S-JEPA downstream protocol: 5-fold within-subject stratified CV.
    # Alternative: set cv_scheme='liu_repeated_holdout' to use repeated 60/40 holdout.
    "cv_scheme": "sjepa_5fold",              # 'sjepa_5fold' | 'liu_repeated_holdout'
    "n_splits":  5,                          # used for sjepa_5fold; 40 trials -> 32 train / 8 test
    "n_repeats": 10,                         # used only for liu_repeated_holdout
    "test_size": 0.40,                       # used only for liu_repeated_holdout

    # --- Embedding ---
    "embedding_hook": "feature_encoder",    # 'feature_encoder' or 'spatial_conv'
    "embedding_pool": "mean",               # 'mean'|'max'|'meanmax' over tokens, or 'flatten'
    "use_pca":        True,
    "pca_max_components": 20,                # kept below train fold size; actual n_comp is clipped

    # --- Classifier ---
    # 'shrinkage_lda' or 'logistic_l2'
    "classifier": "shrinkage_lda",
    "logistic_C":  1.0,

    # --- Spatial conv fine-tuning ---
    # Mirrors the S-JEPA 'new-pre-local' spirit: train only spatial_conv + final_layer per fold.
    # Do not increase epochs until the fixed preprocessing result is known.
    "finetune_spatial_conv": True,
    "finetune_epochs":       5000,
    "finetune_lr":           1e-3,
    "finetune_batch_size":   8,
    # Internal validation is taken only from the outer training fold.
    # For sjepa_5fold on Liu2024: outer train=32 trials -> approx 26 fine-tune train / 6 validation.
    # Set to 0.0 to disable early-stopping validation and train for finetune_epochs on all outer-train trials.
    "finetune_val_split":    0.20,
    "finetune_val_random_state": 2026,
    "finetune_patience":     50,

    # --- Misc ---
    "save_embeddings": True,
    "device": "auto",
}

DATA_ROOT    = Path(CONFIG["data_root"])
ARTIFACT_ROOT = Path(CONFIG["artifact_root"])
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = resolve_device(CONFIG["device"])
SFREQ_RAW   = CONFIG["sfreq_raw"]
SFREQ_MODEL = CONFIG["sfreq_model"]
MI_SAMPLES_RAW   = int((CONFIG["mi_window_s"][1] - CONFIG["mi_window_s"][0]) * SFREQ_RAW)
WINDOW_SAMPLES   = CONFIG["window_samples"]

np.random.seed(CONFIG["random_state"])
random.seed(CONFIG["random_state"])
torch.manual_seed(CONFIG["random_state"])

print(f"Device:       {DEVICE}")
print(f"Data root:    {DATA_ROOT}")
print(f"Artifacts:    {ARTIFACT_ROOT}")
print(f"MI window:    {CONFIG['mi_window_s']} s → {MI_SAMPLES_RAW} samples at {SFREQ_RAW} Hz")
print(f"Model input:  {WINDOW_SAMPLES} samples at {SFREQ_MODEL} Hz")
print(f"CV scheme:    {CONFIG['cv_scheme']}")
if CONFIG["cv_scheme"] == "sjepa_5fold":
    print(f"CV folds:     {CONFIG['n_splits']} folds/subject → approx 32 train / 8 test trials")
else:
    print(f"CV folds:     {CONFIG['n_repeats']} repeats/subject, test_size={CONFIG['test_size']}")
print(f"Classifier:   {CONFIG['classifier']}")


Device:       cpu
Data root:    ../../liu2024_data/liu2024_figshare/sourcedata
Artifacts:    ../../artifacts/liu2024_sjepa_embeddings_lda_sjepa5fold_fixed
MI window:    (1.5, 5.7) s → 2100 samples at 500 Hz
Model input:  537 samples at 128 Hz
CV scheme:    sjepa_5fold
CV folds:     5 folds/subject → approx 32 train / 8 test trials
Classifier:   shrinkage_lda


## 3. Liu2024 Channel Constants

In [3]:
SOURCE_EEG_NAMES_30 = [
    "Fp1", "Fp2", "Fz", "F3", "F4", "F7", "F8", "FCz", "FC3", "FC4",
    "FT7", "FT8", "Cz", "C3", "C4", "T3", "T4", "CPz",
    "CP3", "CP4", "TP7", "TP8", "Pz", "P3", "P4", "T5", "T6", "Oz", "O1", "O2",
]
CPZ_IDX     = 17
EEG_KEEP_IDX = [i for i in range(30) if i != CPZ_IDX]
EEG_NAMES    = [SOURCE_EEG_NAMES_30[i] for i in EEG_KEEP_IDX]
N_CHANS      = len(EEG_KEEP_IDX)  # 29

def make_liu_info(sfreq):
    info = mne.create_info(
        ch_names=EEG_NAMES,
        sfreq=float(sfreq),
        ch_types=["eeg"] * N_CHANS,
    )
    montage = mne.channels.make_standard_montage("standard_1020")
    info.set_montage(montage, match_case=False, on_missing="ignore")
    return info

MNE_INFO = make_liu_info(SFREQ_MODEL)
CHS_INFO = MNE_INFO["chs"]
print(f"N channels: {N_CHANS}, ch_names[:5]: {EEG_NAMES[:5]}")

N channels: 29, ch_names[:5]: ['Fp1', 'Fp2', 'Fz', 'F3', 'F4']


## 4. Data Loading and Preprocessing

In [4]:
def subject_id_from_path(path):
    m = re.search(r"sub[-_ ]?(\d{1,2})", str(path), flags=re.IGNORECASE)
    return int(m.group(1)) if m else int(re.findall(r"\d+", Path(path).stem)[-1])


def preprocess_subject(mat_path):
    """
    Load Liu2024 source .mat → (40, 29, WINDOW_SAMPLES) at SFREQ_MODEL Hz, y (40,).
    Uses MNE RawArray for average reference + resample + bandpass.
    Returns float32 array compatible with S-JEPA PreLocal.
    """
    mat = loadmat(str(mat_path), squeeze_me=True, struct_as_record=False)
    raw_data = mat.get("rawdata", mat.get("data", None))
    labels   = mat.get("labels", mat.get("label", None))
    # Liu2024 figshare files nest arrays under an 'eeg' struct: eeg.rawdata / eeg.label
    if raw_data is None or labels is None:
        for k, v in mat.items():
            if k.startswith("__"):
                continue
            if hasattr(v, "_fieldnames"):
                if raw_data is None and "rawdata" in v._fieldnames:
                    raw_data = getattr(v, "rawdata")
                if labels is None and "label" in v._fieldnames:
                    labels = getattr(v, "label")
            elif raw_data is None and isinstance(v, np.ndarray) and v.ndim == 3:
                raw_data = v

    raw_data = np.asarray(raw_data, dtype=np.float64)
    # Normalise to trials x 33 x 4000
    trial_ax = next(ax for ax, sz in enumerate(raw_data.shape) if sz == 40)
    raw_data = np.moveaxis(raw_data, trial_ax, 0)
    if raw_data.shape[2] == 33:
        raw_data = raw_data.transpose(0, 2, 1)  # wait — channels must be dim 1
    # After moveaxis, shape should be 40 x 33 x 4000 OR 40 x 4000 x 33
    if raw_data.shape[1] != 33 and raw_data.shape[2] == 33:
        raw_data = raw_data.transpose(0, 2, 1)
    assert raw_data.shape == (40, 33, 4000), f"Shape mismatch: {raw_data.shape}"

    y = np.asarray(labels, dtype=int).ravel()
    if set(np.unique(y).tolist()).issubset({1, 2}):
        y = y - 1
    assert len(y) == 40, f"Expected 40 labels, got {len(y)}"
    assert set(np.unique(y).tolist()).issubset({0, 1}), f"Unexpected labels: {np.unique(y)}"

    # Select 29 EEG channels
    eeg = raw_data[:, EEG_KEEP_IDX, :].astype(np.float64)  # 40 x 29 x 4000

    # Convert to MNE RawArray per subject by concatenating trials as continuous signal.
    # IMPORTANT: eeg is trials x channels x time. We must reorder to channels x trials x time
    # before flattening. A direct eeg.reshape(n_ch, n_trials*n_t) scrambles channels/trials.
    n_trials, n_ch, n_t = eeg.shape
    continuous = eeg.transpose(1, 0, 2).reshape(n_ch, n_trials * n_t) * 1e-6  # → Volts for MNE
    info_raw = mne.create_info(ch_names=EEG_NAMES, sfreq=float(SFREQ_RAW), ch_types=["eeg"] * N_CHANS)
    raw_mne = mne.io.RawArray(continuous, info_raw, verbose=False)

    # Average reference
    raw_mne.set_eeg_reference("average", projection=False, verbose=False)
    # Resample to 128 Hz
    raw_mne.resample(SFREQ_MODEL, npad="auto", verbose=False)
    # Bandpass 0.5–40 Hz
    bp_low, bp_high = CONFIG["bandpass_hz"]
    raw_mne.filter(bp_low, bp_high, method="fir", phase="zero", verbose=False)

    # Reshape back to trials. This is the inverse of transpose(1, 0, 2).reshape(...).
    data_resampled = raw_mne.get_data() * 1e6  # back to microvolts
    n_t_resampled = int(n_t * SFREQ_MODEL / SFREQ_RAW)
    assert data_resampled.shape[1] == n_trials * n_t_resampled, (
        f"Unexpected resampled length: {data_resampled.shape[1]} vs {n_trials*n_t_resampled}"
    )
    data_trials = data_resampled.reshape(N_CHANS, n_trials, n_t_resampled)
    data_trials = data_trials.transpose(1, 0, 2)  # → 40 x 29 x n_t_resampled

    # Extract MI window and crop to WINDOW_SAMPLES
    mi_start_s = CONFIG["mi_window_s"][0]
    mi_start_idx = int(mi_start_s * SFREQ_MODEL)
    mi_stop_idx  = mi_start_idx + WINDOW_SAMPLES
    assert mi_stop_idx <= n_t_resampled, (
        f"Window [{mi_start_idx}:{mi_stop_idx}] exceeds resampled length {n_t_resampled}"
    )
    X = data_trials[:, :, mi_start_idx:mi_stop_idx].astype(np.float32)  # 40 x 29 x 537

    assert X.shape == (40, N_CHANS, WINDOW_SAMPLES), f"Final X shape mismatch: {X.shape}"
    assert np.isfinite(X).all(), "Non-finite values in preprocessed data"
    return X, y


def find_mat_files(root):
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"Data root not found: {root}")
    return sorted(root.rglob("*.mat"))


mat_files  = find_mat_files(DATA_ROOT)
all_sids   = sorted({subject_id_from_path(f) for f in mat_files})
SUBJECT_IDS = all_sids if CONFIG["subjects"] == "all" else sorted(int(s) for s in CONFIG["subjects"])
sid_to_path = {subject_id_from_path(f): f for f in mat_files if subject_id_from_path(f) in SUBJECT_IDS}

print(f"Found {len(mat_files)} .mat files, using subjects: {SUBJECT_IDS[:5]}...")

Found 50 .mat files, using subjects: [1, 2, 3, 4, 5]...


## 5. S-JEPA Model Loading

In [5]:
NEW_LAYER_PREFIXES = ("spatial_conv.", "final_layer.")


def load_sjepa_model(cfg, n_chans, chs_info, n_times, n_outputs=2):
    """
    Load SignalJEPA_PreLocal from HuggingFace or a local checkpoint.
    Returns the model with all weights frozen except spatial_conv + final_layer.
    """
    kwargs = dict(
        n_chans=n_chans,
        chs_info=chs_info,
        n_times=n_times,
        n_outputs=n_outputs,
    )
    ckpt = cfg.get("sjepa_checkpoint_path")
    if ckpt is not None:
        print(f"Loading S-JEPA from local checkpoint: {ckpt}")
        model = SignalJEPA_PreLocal(**kwargs)
        state = torch.load(ckpt, map_location="cpu")
        missing, unexpected = model.load_state_dict(state, strict=False)
        print(f"  Missing keys: {missing[:5]}{'...' if len(missing) > 5 else ''}")
        print(f"  Unexpected keys: {unexpected[:5]}{'...' if len(unexpected) > 5 else ''}")
    else:
        print(f"Loading S-JEPA from HuggingFace: {cfg['sjepa_repo_id']}")
        model = SignalJEPA_PreLocal.from_pretrained(
            cfg["sjepa_repo_id"], **kwargs, strict=False
        )

    # Freeze everything
    for p in model.parameters():
        p.requires_grad = False

    # Unfreeze spatial_conv and final_layer
    for name, p in model.named_parameters():
        if any(name.startswith(prefix) for prefix in NEW_LAYER_PREFIXES):
            p.requires_grad = True

    total      = sum(p.numel() for p in model.parameters())
    trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Total params:     {total:,}")
    print(f"  Trainable params: {trainable:,} (spatial_conv + final_layer)")
    return model


# Load once; we'll reset the trainable weights per fold
BASE_MODEL = load_sjepa_model(CONFIG, N_CHANS, CHS_INFO, WINDOW_SAMPLES)
BASE_MODEL = BASE_MODEL.to(DEVICE)
print(f"Model on: {next(BASE_MODEL.parameters()).device}")

Loading S-JEPA from HuggingFace: braindecode/signal-jepa_without-chans


  Total params:     16,010
  Trainable params: 2,170 (spatial_conv + final_layer)
Model on: cpu


## 6. Spatial Conv Fine-tuning + Embedding Extraction

For each fold we:
1. Deep-copy the base model and reset spatial_conv + final_layer weights.
2. Fine-tune spatial_conv + final_layer on training trials (cross-entropy, early stopping on a held-out 20%).
3. Extract embeddings from the trained model for all train + test trials.

The embedding is the mean-pooled token sequence after the local encoder and spatial aggregation,
**before** the final classification layer. This gives a compact representation per trial.

In [6]:
import copy


class TrialDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(np.asarray(X, dtype=np.float32))
        self.y = torch.from_numpy(np.asarray(y, dtype=np.int64))
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def make_internal_train_val_indices(y_train, cfg):
    """
    Build a balanced internal validation split from the outer training fold.

    This validation split is used only for early stopping / best-checkpoint selection
    during spatial-conv fine-tuning. The outer test fold is never touched here.
    """
    y_train = np.asarray(y_train, dtype=np.int64)
    n = len(y_train)
    val_split = float(cfg.get("finetune_val_split", 0.20))

    if val_split <= 0.0:
        return np.arange(n), None

    rng = np.random.default_rng(int(cfg.get("finetune_val_random_state", cfg.get("random_state", 2026))))
    train_parts = []
    val_parts = []

    for cls in np.unique(y_train):
        cls_idx = np.where(y_train == cls)[0]
        cls_idx = rng.permutation(cls_idx)

        # Keep at least 1 validation trial per class, but never consume the whole class.
        n_val_cls = int(round(len(cls_idx) * val_split))
        n_val_cls = max(1, n_val_cls)
        n_val_cls = min(n_val_cls, len(cls_idx) - 1)

        val_parts.append(cls_idx[:n_val_cls])
        train_parts.append(cls_idx[n_val_cls:])

    tr_idx = np.concatenate(train_parts)
    val_idx = np.concatenate(val_parts)
    tr_idx = rng.permutation(tr_idx)
    val_idx = rng.permutation(val_idx)
    return tr_idx, val_idx


def finetune_spatial_conv(model, X_train, y_train, cfg, device):
    """
    Fine-tune spatial_conv + final_layer on the outer training fold only.

    If cfg['finetune_val_split'] > 0, uses a small balanced internal validation split
    for early stopping and restores the best validation-loss checkpoint. If set to 0,
    trains for cfg['finetune_epochs'] on all outer-train trials and returns the final model.

    Returns:
      model, summary
    """
    model = copy.deepcopy(model)
    # Re-init trainable weights to avoid carry-over between subjects
    for name, module in model.named_modules():
        if any(name.startswith(pf.rstrip(".")) for pf in NEW_LAYER_PREFIXES):
            if hasattr(module, "reset_parameters"):
                module.reset_parameters()

    model.train()
    optimizer = optim.Adam(
        [p for p in model.parameters() if p.requires_grad],
        lr=cfg["finetune_lr"],
        weight_decay=1e-4,
    )
    criterion = nn.CrossEntropyLoss()

    tr_idx, val_idx = make_internal_train_val_indices(y_train, cfg)

    ds_tr = TrialDataset(X_train[tr_idx], y_train[tr_idx])
    dl_tr = torch.utils.data.DataLoader(
        ds_tr,
        batch_size=cfg["finetune_batch_size"],
        shuffle=True,
    )

    use_validation = val_idx is not None and len(val_idx) > 0
    if use_validation:
        ds_val = TrialDataset(X_train[val_idx], y_train[val_idx])
        dl_val = torch.utils.data.DataLoader(
            ds_val,
            batch_size=cfg["finetune_batch_size"],
            shuffle=False,
        )
    else:
        dl_val = None

    best_val_loss = float("inf")
    best_state = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
    patience_ctr = 0
    epochs_ran = 0
    last_train_loss = float("nan")
    last_val_loss = float("nan")

    for epoch in range(int(cfg["finetune_epochs"])):
        epochs_ran = epoch + 1
        model.train()
        train_loss = 0.0
        for xb, yb in dl_tr:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            train_loss += float(loss.item())
        last_train_loss = train_loss / max(len(dl_tr), 1)

        if not use_validation:
            continue

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in dl_val:
                xb, yb = xb.to(device), yb.to(device)
                out = model(xb)
                val_loss += float(criterion(out, yb).item())
        val_loss /= max(len(dl_val), 1)
        last_val_loss = val_loss

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_state = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= int(cfg["finetune_patience"]):
                break

    if use_validation:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
        selected_checkpoint = "best_internal_validation_loss"
    else:
        selected_checkpoint = "final_epoch_no_internal_validation"

    model.eval()
    summary = {
        "finetune_use_validation": bool(use_validation),
        "finetune_val_split": float(cfg.get("finetune_val_split", 0.20)),
        "finetune_n_train_inner": int(len(tr_idx)),
        "finetune_n_val_inner": int(0 if val_idx is None else len(val_idx)),
        "finetune_train_counts_inner": np.bincount(y_train[tr_idx], minlength=2).astype(int).tolist(),
        "finetune_val_counts_inner": None if val_idx is None else np.bincount(y_train[val_idx], minlength=2).astype(int).tolist(),
        "finetune_epochs_ran": int(epochs_ran),
        "finetune_best_val_loss": None if not use_validation else float(best_val_loss),
        "finetune_last_train_loss": float(last_train_loss),
        "finetune_last_val_loss": None if not use_validation else float(last_val_loss),
        "finetune_selected_checkpoint": selected_checkpoint,
    }
    return model, summary


@torch.no_grad()
def extract_embeddings(model, X, cfg, device):
    """
    Extract per-trial RICH embeddings from S-JEPA PreLocal via a forward hook.

    PreLocal forward is: spatial_conv -> feature_encoder -> final_layer (2-D head).
    We hook an intermediate module and pool to a fixed (n_trials, D) matrix:
      cfg['embedding_hook']:
        'feature_encoder' (default): rich local-token tensor (B, n_tokens, emb_dim) -> pool over tokens
        'spatial_conv'             : spatially-filtered signal (B, n_spat_filters, n_times) -> pool over time
      cfg['embedding_pool']:
        'mean' | 'max' | 'meanmax' over the pooled axis, or 'flatten' (full flatten).
    Returns: numpy (n_trials, D).  (D probed at runtime; no longer the 2-D logits.)
    """
    model.eval()
    pool = cfg.get("embedding_pool", "mean")
    hook_name = cfg.get("embedding_hook", "feature_encoder")
    target = getattr(model, hook_name, None)
    if target is None:
        raise AttributeError(f"model has no submodule '{hook_name}' to hook")

    captured = {}
    def _hook(module, inp, out):
        captured["z"] = out.detach()
    handle = target.register_forward_hook(_hook)

    ds = TrialDataset(X, np.zeros(len(X), dtype=np.int64))
    dl = torch.utils.data.DataLoader(ds, batch_size=32, shuffle=False)
    all_embs = []
    try:
        for xb, _ in dl:
            xb = xb.to(device)
            _ = model(xb)                      # full forward; hook captures the intermediate
            z = captured["z"]
            if z.dim() == 2:
                feat = z
            else:
                # feature_encoder -> pool over tokens (dim 1); spatial_conv -> pool over time (dim 2)
                axis = 2 if hook_name == "spatial_conv" else 1
                if pool == "flatten":
                    feat = z.flatten(start_dim=1)
                elif pool == "max":
                    feat = z.max(dim=axis).values.flatten(start_dim=1)
                elif pool == "meanmax":
                    feat = torch.cat([z.mean(dim=axis), z.max(dim=axis).values], dim=-1).flatten(start_dim=1)
                else:  # mean
                    feat = z.mean(dim=axis).flatten(start_dim=1)
            all_embs.append(feat.cpu().numpy())
    finally:
        handle.remove()

    embeddings = np.concatenate(all_embs, axis=0).astype(np.float32)
    if not hasattr(extract_embeddings, "_printed"):
        print(f"  [hook={hook_name} pool={pool}] embedding matrix: {embeddings.shape}")
        extract_embeddings._printed = True
    return embeddings


print("Fine-tuning and embedding extraction functions defined.")

Fine-tuning and embedding extraction functions defined.


> **Embedding dimensionality (resolved).** `extract_embeddings` now registers a forward hook on an
> intermediate S-JEPA module instead of using the 2-D class logits. With `embedding_hook="feature_encoder"`
> (default) it captures the rich local-token tensor `(batch, n_tokens, emb_dim=64)` and pools over the token
> axis (`embedding_pool` = `mean`/`max`/`meanmax`/`flatten`) to a fixed per-trial vector (D = 64 for `mean`,
> ~1024 for `flatten`). `embedding_hook="spatial_conv"` instead captures the spatially-filtered signal.
> This is the meaningful representation the LDA probe and the hybrid's S-JEPA branch need; the old 2-D-logit
> path is gone. Leakage is unchanged: spatial_conv fine-tune, PCA, and LDA are still fit on the train fold only.


## 7. Classifier

In [7]:
def make_clf(cfg):
    if cfg["classifier"] == "shrinkage_lda":
        return LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
    elif cfg["classifier"] == "logistic_l2":
        return LogisticRegression(C=cfg["logistic_C"], penalty="l2",
                                  solver="lbfgs", max_iter=500)
    else:
        raise ValueError(f"Unknown classifier: {cfg['classifier']}")


def collapse_diagnostics(y_pred, n_classes=2):
    counts = np.bincount(y_pred, minlength=n_classes)
    dominant = counts.max() / counts.sum() if counts.sum() > 0 else 1.0
    return {
        "collapse_flag":  bool(dominant > 0.95),
        "collapse_ratio": float(dominant),
        "pred_counts":    counts.tolist(),
    }


print("Classifier helpers defined.")

Classifier helpers defined.


## 8. Per-Subject Cross-Validation Runner

Default mode is now `sjepa_5fold`, which uses `StratifiedKFold(n_splits=5, shuffle=True)` inside each subject. With Liu2024's 40 balanced trials, each fold should have 32 train and 8 test trials. The old repeated 60/40 Liu-style holdout remains available with `cv_scheme="liu_repeated_holdout"`.


Internal validation is nested inside the outer training fold only. With `finetune_val_split=0.20`, each 5-fold subject split uses about 26 trials for spatial-conv fine-tuning, 6 trials for early-stopping validation, and 8 untouched trials for testing.


In [8]:
def make_subject_splits(X, y, cfg):
    """Return iterable of (fold_idx, train_idx, test_idx) for one subject."""
    scheme = cfg.get("cv_scheme", "sjepa_5fold")
    if scheme == "sjepa_5fold":
        splitter = StratifiedKFold(
            n_splits=cfg["n_splits"],
            shuffle=True,
            random_state=cfg["random_state"],
        )
    elif scheme == "liu_repeated_holdout":
        splitter = StratifiedShuffleSplit(
            n_splits=cfg["n_repeats"],
            test_size=cfg["test_size"],
            random_state=cfg["random_state"],
        )
    else:
        raise ValueError(f"Unknown cv_scheme: {scheme}")

    for fold_idx, (tr_idx, te_idx) in enumerate(splitter.split(X, y)):
        yield fold_idx, tr_idx, te_idx


def run_subject(sid, X, y, cfg, base_model, device):
    fold_results = []

    for fold_idx, tr_idx, te_idx in make_subject_splits(X, y, cfg):
        X_train, X_test = X[tr_idx], X[te_idx]
        y_train, y_test = y[tr_idx], y[te_idx]

        assert len(np.unique(y_train)) == 2, f"Sub {sid} fold {fold_idx}: single class in train"
        assert len(np.unique(y_test)) == 2, f"Sub {sid} fold {fold_idx}: single class in test"
        assert np.isfinite(X_train).all() and np.isfinite(X_test).all()

        emb_train = None
        emb_test = None
        train_pred = None
        y_pred = None
        pca_n_components = 0
        pca_var_sum = float("nan")
        finetune_summary = {}

        try:
            # --- Step 1: Fine-tune spatial_conv on training fold only ---
            if cfg["finetune_spatial_conv"]:
                model, finetune_summary = finetune_spatial_conv(base_model, X_train, y_train, cfg, device)
            else:
                model = copy.deepcopy(base_model)
                model.eval()
                finetune_summary = {
                    "finetune_use_validation": False,
                    "finetune_n_train_inner": int(len(y_train)),
                    "finetune_n_val_inner": 0,
                    "finetune_train_counts_inner": np.bincount(y_train, minlength=2).astype(int).tolist(),
                    "finetune_val_counts_inner": None,
                    "finetune_epochs_ran": 0,
                    "finetune_selected_checkpoint": "no_spatial_finetune",
                }

            # --- Step 2: Extract embeddings ---
            emb_train = extract_embeddings(model, X_train, cfg, device)  # (n_train, D)
            emb_test  = extract_embeddings(model, X_test,  cfg, device)  # (n_test, D)

            # --- Step 3: PCA fit on train only ---
            if cfg["use_pca"] and emb_train.shape[1] > cfg["pca_max_components"]:
                n_comp = min(cfg["pca_max_components"], emb_train.shape[0] - 1)
                pca = PCA(n_components=n_comp, random_state=cfg["random_state"])
                emb_train = pca.fit_transform(emb_train)
                emb_test  = pca.transform(emb_test)
                pca_n_components = int(n_comp)
                pca_var_sum = float(np.sum(pca.explained_variance_ratio_))
            else:
                pca_n_components = int(emb_train.shape[1])

            # --- Step 4: Classify ---
            clf = make_clf(cfg)
            clf.fit(emb_train, y_train)
            train_pred = clf.predict(emb_train)
            y_pred = clf.predict(emb_test)

        except Exception as exc:
            print(f"  Sub {sid} fold {fold_idx} ERROR: {exc}")
            train_pred = np.zeros(len(y_train), dtype=int)
            y_pred = np.zeros(len(y_test), dtype=int)

        train_acc  = float(accuracy_score(y_train, train_pred))
        train_bacc = float(balanced_accuracy_score(y_train, train_pred))
        acc  = float(accuracy_score(y_test, y_pred))
        bacc = float(balanced_accuracy_score(y_test, y_pred))
        cm   = confusion_matrix(y_test, y_pred, labels=[0, 1]).tolist()
        diag = collapse_diagnostics(y_pred)

        cm_arr = np.array(cm)
        left_recall  = cm_arr[0, 0] / cm_arr[0].sum() if cm_arr[0].sum() > 0 else float("nan")
        right_recall = cm_arr[1, 1] / cm_arr[1].sum() if cm_arr[1].sum() > 0 else float("nan")

        fold_results.append({
            "subject_id":        int(sid),
            "fold_id":           int(fold_idx),
            "cv_scheme":         cfg.get("cv_scheme", "sjepa_5fold"),
            "accuracy":          acc,
            "balanced_accuracy": bacc,
            "train_accuracy":    train_acc,
            "train_balanced_accuracy": train_bacc,
            "left_recall":       float(left_recall),
            "right_recall":      float(right_recall),
            "confusion_matrix":  cm,
            "collapse_flag":     diag["collapse_flag"],
            "collapse_ratio":    diag["collapse_ratio"],
            "pred_counts":       diag["pred_counts"],
            "y_train_counts":    np.bincount(y_train, minlength=2).astype(int).tolist(),
            "y_test_counts":     np.bincount(y_test, minlength=2).astype(int).tolist(),
            "n_train":           int(len(y_train)),
            "n_test":            int(len(y_test)),
            "embedding_dim":     int(emb_train.shape[1]) if emb_train is not None else -1,
            "pca_n_components":  int(pca_n_components),
            "pca_explained_variance_sum": float(pca_var_sum),
            **finetune_summary,
        })

    return fold_results


print("Subject runner defined.")


Subject runner defined.


## 9. Run All Subjects

In [9]:
ALL_FOLD_RESULTS  = []
SUBJECT_SUMMARIES = []
SAVED_EMBEDDINGS  = {}  # {sid: {"X_train": ..., "X_test": ..., "y": ...}} if save_embeddings

print(f"Running {len(SUBJECT_IDS)} subjects | cv={CONFIG['cv_scheme']} | clf={CONFIG['classifier']}")
print("=" * 60)

for sid in SUBJECT_IDS:
    mat_path = sid_to_path.get(sid)
    if mat_path is None:
        print(f"  Sub {sid:02d}: no file found, skipping")
        continue
    try:
        X, y = preprocess_subject(mat_path)
    except Exception as exc:
        print(f"  Sub {sid:02d}: preprocess error — {exc}")
        continue

    fold_res = run_subject(sid, X, y, CONFIG, BASE_MODEL, DEVICE)
    ALL_FOLD_RESULTS.extend(fold_res)

    accs  = [r["accuracy"] for r in fold_res]
    baccs = [r["balanced_accuracy"] for r in fold_res]
    collapses = sum(1 for r in fold_res if r["collapse_flag"])
    mean_bacc = np.mean(baccs)

    SUBJECT_SUMMARIES.append({
        "subject_id":             sid,
        "mean_accuracy":          float(np.mean(accs)),
        "std_accuracy":           float(np.std(accs)),
        "mean_balanced_accuracy": float(mean_bacc),
        "std_balanced_accuracy":  float(np.std(baccs)),
        "n_folds":                len(fold_res),
        "n_collapsed_folds":      collapses,
    })

    print(f"  Sub {sid:02d}: bal_acc={mean_bacc*100:.1f}% ± {np.std(baccs)*100:.1f}%  "
          f"collapse={collapses}/{len(fold_res)}")

print("=" * 60)
print(f"Done. Total folds: {len(ALL_FOLD_RESULTS)}")

Running 50 subjects | cv=sjepa_5fold | clf=shrinkage_lda
  [hook=feature_encoder pool=mean] embedding matrix: (32, 64)
  Sub 01: bal_acc=55.0% ± 10.0%  collapse=1/5
  Sub 02: bal_acc=42.5% ± 10.0%  collapse=0/5
  Sub 03: bal_acc=62.5% ± 11.2%  collapse=0/5
  Sub 04: bal_acc=47.5% ± 12.2%  collapse=1/5
  Sub 05: bal_acc=42.5% ± 17.0%  collapse=0/5
  Sub 06: bal_acc=37.5% ± 20.9%  collapse=1/5
  Sub 07: bal_acc=42.5% ± 12.7%  collapse=0/5
  Sub 08: bal_acc=60.0% ± 5.0%  collapse=0/5
  Sub 09: bal_acc=42.5% ± 12.7%  collapse=0/5
  Sub 10: bal_acc=50.0% ± 13.7%  collapse=0/5
  Sub 11: bal_acc=37.5% ± 15.8%  collapse=0/5
  Sub 12: bal_acc=55.0% ± 10.0%  collapse=0/5
  Sub 13: bal_acc=70.0% ± 12.7%  collapse=0/5
  Sub 14: bal_acc=57.5% ± 12.7%  collapse=0/5
  Sub 15: bal_acc=62.5% ± 13.7%  collapse=0/5
  Sub 16: bal_acc=30.0% ± 12.7%  collapse=0/5
  Sub 17: bal_acc=42.5% ± 12.7%  collapse=0/5
  Sub 18: bal_acc=62.5% ± 13.7%  collapse=0/5
  Sub 19: bal_acc=45.0% ± 6.1%  collapse=0/5
  Sub 20:

In [10]:
# --- Cache FROZEN rich embeddings per subject (for the TWFB hybrid, Branch A) ---
# Deterministic, label-free: uses the frozen pretrained BASE_MODEL (no per-fold fine-tune),
# so each trial maps to one fixed embedding the hybrid can load directly.
if CONFIG.get("save_embeddings", False):
    emb_dir = ARTIFACT_ROOT / "embeddings"
    emb_dir.mkdir(parents=True, exist_ok=True)
    frozen = copy.deepcopy(BASE_MODEL).to(DEVICE).eval()
    manifest = {"hook": CONFIG.get("embedding_hook", "feature_encoder"),
                "pool": CONFIG.get("embedding_pool", "mean"),
                "window_samples": WINDOW_SAMPLES, "sfreq_model": SFREQ_MODEL,
                "mi_window_s": list(CONFIG["mi_window_s"]), "subjects": []}
    n_saved = 0
    for sid in SUBJECT_IDS:
        mat_path = sid_to_path.get(sid)
        if mat_path is None:
            continue
        try:
            Xs, ys = preprocess_subject(mat_path)
            Es = extract_embeddings(frozen, Xs, CONFIG, DEVICE)   # (40, D), frozen
            np.savez(emb_dir / f"sub-{sid:02d}.npz", X=Es, y=ys)
            manifest["subjects"].append(int(sid)); manifest["embedding_dim"] = int(Es.shape[1])
            n_saved += 1
        except Exception as exc:
            print(f"  Sub {sid:02d}: embedding cache error — {exc}")
    json.dump(manifest, open(emb_dir / "manifest.json", "w"), indent=2)
    print(f"Cached frozen embeddings for {n_saved} subjects (D={manifest.get('embedding_dim')}) -> {emb_dir}")
else:
    print("save_embeddings=False -> skipping frozen embedding cache")

Cached frozen embeddings for 50 subjects (D=64) -> ../../artifacts/liu2024_sjepa_embeddings_lda_sjepa5fold_fixed/embeddings


## 10. Aggregate Results

In [11]:

if not ALL_FOLD_RESULTS:
    raise RuntimeError(
        "ALL_FOLD_RESULTS is empty — no folds completed successfully. "
        "Check that subjects loaded and that the run cell (Section 9) executed without errors."
    )

fold_df    = pd.DataFrame(ALL_FOLD_RESULTS)
subject_df = pd.DataFrame(SUBJECT_SUMMARIES)

all_baccs = fold_df["balanced_accuracy"].values
all_accs  = fold_df["accuracy"].values
n_collapsed = int(fold_df["collapse_flag"].sum())

cm_total = np.zeros((2, 2), dtype=int)
for row in ALL_FOLD_RESULTS:
    cm_total += np.array(row["confusion_matrix"])

left_recall_agg  = cm_total[0, 0] / cm_total[0].sum() if cm_total[0].sum() > 0 else float("nan")
right_recall_agg = cm_total[1, 1] / cm_total[1].sum() if cm_total[1].sum() > 0 else float("nan")

global_summary = {
    "method":                 f"sjepa_embeddings_{CONFIG['classifier']}",
    "classifier":             CONFIG["classifier"],
    "cv_scheme":              CONFIG.get("cv_scheme", "sjepa_5fold"),
    "n_splits":               CONFIG.get("n_splits"),
    "n_repeats":              CONFIG.get("n_repeats"),
    "test_size":              CONFIG.get("test_size"),
    "finetune_spatial_conv":  CONFIG["finetune_spatial_conv"],
    "n_subjects":             len(SUBJECT_SUMMARIES),
    "n_folds_total":          len(ALL_FOLD_RESULTS),
    "mean_accuracy":          float(np.mean(all_accs)),
    "std_accuracy":           float(np.std(all_accs)),
    "mean_balanced_accuracy": float(np.mean(all_baccs)),
    "std_balanced_accuracy":  float(np.std(all_baccs)),
    "n_collapsed_folds":      n_collapsed,
    "collapse_rate":          float(n_collapsed / max(len(ALL_FOLD_RESULTS), 1)),
    "left_recall_agg":        float(left_recall_agg),
    "right_recall_agg":       float(right_recall_agg),
    "confusion_matrix":       cm_total.tolist(),
}

print("=" * 60)
print(f"GLOBAL RESULTS — S-JEPA embeddings + {CONFIG['classifier']}")
print(f"  CV scheme:              {global_summary['cv_scheme']}")
print(f"  Mean Balanced Accuracy: {global_summary['mean_balanced_accuracy']*100:.2f}% ± {global_summary['std_balanced_accuracy']*100:.2f}%")
print(f"  Mean Accuracy:          {global_summary['mean_accuracy']*100:.2f}% ± {global_summary['std_accuracy']*100:.2f}%")
print(f"  Left recall:  {left_recall_agg*100:.1f}%  |  Right recall: {right_recall_agg*100:.1f}%")
print(f"  Collapsed folds: {n_collapsed}/{len(ALL_FOLD_RESULTS)} ({global_summary['collapse_rate']*100:.1f}%)")
print(f"  Aggregated CM: {cm_total.tolist()}")
print("=" * 60)


GLOBAL RESULTS — S-JEPA embeddings + shrinkage_lda
  CV scheme:              sjepa_5fold
  Mean Balanced Accuracy: 52.55% ± 17.65%
  Mean Accuracy:          52.55% ± 17.65%
  Left recall:  53.0%  |  Right recall: 52.1%
  Collapsed folds: 3/250 (1.2%)
  Aggregated CM: [[530, 470], [479, 521]]


## 11. Save Artifacts

In [12]:
fold_df.to_csv(ARTIFACT_ROOT / "fold_results.csv", index=False)
subject_df.to_csv(ARTIFACT_ROOT / "subject_summary.csv", index=False)
with open(ARTIFACT_ROOT / "global_summary.json", "w") as f:
    json.dump(global_summary, f, indent=2)
with open(ARTIFACT_ROOT / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2, default=str)

print(f"Saved: fold_results.csv, subject_summary.csv, global_summary.json, config.json")

Saved: fold_results.csv, subject_summary.csv, global_summary.json, config.json


## 12. Plots

In [13]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(4, 3))
im = ax.imshow(cm_total, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Pred Left", "Pred Right"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["True Left", "True Right"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm_total[i, j]), ha="center", va="center", fontsize=12)
ax.set_title(f"Aggregated CM — S-JEPA + {CONFIG['classifier']} ({CONFIG['cv_scheme']})")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(ARTIFACT_ROOT / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

# Per-subject accuracy
fig, ax = plt.subplots(figsize=(max(8, len(subject_df) * 0.4), 4))
sids  = subject_df["subject_id"].values
baccs = subject_df["mean_balanced_accuracy"].values * 100
stds  = subject_df["std_balanced_accuracy"].values * 100
ax.bar(sids, baccs, yerr=stds, capsize=3, color="mediumpurple", alpha=0.8)
ax.axhline(50, color="red", linestyle="--", label="Chance")
ax.axhline(float(np.mean(baccs)), color="orange", linestyle="-", label=f"Mean={np.mean(baccs):.1f}%")
ax.set_xlabel("Subject ID")
ax.set_ylabel("Balanced Accuracy (%)")
ax.set_title(f"Per-subject — S-JEPA + {CONFIG['classifier']} ({CONFIG['cv_scheme']})")
ax.legend()
ax.set_xticks(sids)
ax.set_xticklabels(sids, rotation=90, fontsize=7)
plt.tight_layout()
plt.savefig(ARTIFACT_ROOT / "subject_accuracy_plot.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plots saved.")

Plots saved.


/tmp/ipykernel_1075350/3266832774.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_1075350/3266832774.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
